# In-Browser FPGA Bitstream Manager
Load and program Vivado `.bit` files into the **Zynq XC7Z010 Programmable Logic (PL)** directly from Jupyter Notebook without terminal commands.

### Why not standard PYNQ `Overlay()`?
Standard PYNQ `Overlay()` attempts to communicate through low-level PCAP drivers that can hang or crash the XC7Z010 when loading custom bitstreams. This notebook uses the robust Linux kernel `fpga_manager` interface (`/sys/class/fpga_manager/fpga0/firmware`), which performs the required 16-bit word byte-swap (`conv=swab`) in memory and guarantees a clean programming sequence.



In [ ]:
import os
import subprocess
import time

def program_fpga(bitfile_path):
    """
    Byte-swaps a Vivado .bit file and streams it to Linux fpga_manager.
    """
    if not os.path.isfile(bitfile_path):
        raise FileNotFoundError(f"Bitstream file not found: {bitfile_path}")
        
    basename = os.path.splitext(os.path.basename(bitfile_path))[0]
    swapped_bin_path = f"/tmp/{basename}_swapped.bin"
    firmware_dest = f"/lib/firmware/{basename}_swapped.bin"
    
    print(f"[*] Reading '{bitfile_path}' ({os.path.getsize(bitfile_path):,} bytes)...")
    
    # 1. Perform 16-bit word byte swap in memory
    with open(bitfile_path, "rb") as fin, open(swapped_bin_path, "wb") as fout:
        while chunk := fin.read(65536):
            if len(chunk) % 2 != 0:
                chunk += b'\x00'
            swapped = bytearray(chunk)
            for i in range(0, len(swapped), 2):
                swapped[i], swapped[i+1] = swapped[i+1], swapped[i]
            fout.write(swapped)
            
    print("[*] Byte-swap complete. Copying to /lib/firmware/...")
    subprocess.run(["sudo", "mkdir", "-p", "/lib/firmware"], check=True)
    subprocess.run(["sudo", "cp", swapped_bin_path, firmware_dest], check=True)
    
    # 2. Trigger Linux fpga_manager
    print("[*] Streaming to /sys/class/fpga_manager/fpga0/firmware...")
    cmd = f'echo "{basename}_swapped.bin" | sudo tee /sys/class/fpga_manager/fpga0/firmware'
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
    
    # Clean up tmp file
    if os.path.exists(swapped_bin_path):
        os.remove(swapped_bin_path)
        
    time.sleep(0.2)
    
    # 3. Verify FPGA Manager state
    with open("/sys/class/fpga_manager/fpga0/state", "r") as f:
        state = f.read().strip()
        
    if state == "operating":
        print(f"\n[+] SUCCESS: FPGA is PROGRAMMED and OPERATING! (State: {state})")
        print("[+] Check board: The D1 Green LED (FPGA_DONE) should now be lit solid.")
        return True
    else:
        print(f"\n[-] WARNING: Unexpected FPGA manager state: {state}")
        return False



In [ ]:
# Example: Program your bitstream
# Upload your .bit file to /home/xilinx/ in Jupyter, then set its path here:

BITFILE_PATH = "/home/xilinx/my_design.bit"

if os.path.exists(BITFILE_PATH):
    program_fpga(BITFILE_PATH)
else:
    print(f"Upload your Vivado .bit file to '{BITFILE_PATH}' and re-run this cell.")



In [ ]:
# Current FPGA State Inspector
with open("/sys/class/fpga_manager/fpga0/state", "r") as f:
    print("FPGA Manager State:", f.read().strip())

